In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PowerTransformer
import pingouin as pg

import patsy

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings('ignore')


In [2]:
# 读取数据
# 读取所有sheet到字典
df = pd.read_excel(
    r'E:\pycharm all files\眼动数据处理\EEG混合线性模型分析\按session分开的_eeg&eye&scl&ppg+eeg_ratio+eeg_de.xlsx',
)

df = df.drop(columns=[
    'SCL_mean_yj',
    'AOI转换次数',
    '静态注释熵(SGE)',
    '眼跳注视熵(GTE)',
    'HR',
    'SDNN',
    'RMSSD',
    'AMP'
])
df

,组别,受试者,session,EEG_beta_high_Cz_mean,EEG_beta_high_Fz_mean,EEG_beta_high_Pz_mean,EEG_beta_low_Cz_mean,EEG_beta_low_Fz_mean,EEG_beta_low_Pz_mean,Cz_alpVShi,...,Pz_alpVShi,Pz_alpVSlow,Pz_theVShi,Pz_theVSlow,DE_Cz_beta_high,DE_Cz_beta_low,DE_Fz_beta_high,DE_Fz_beta_low,DE_Pz_beta_high,DE_Pz_beta_low
0,Alcohol,付瑞晗,1,-0.729108,-0.440792,1.180175,-0.683908,-0.657176,1.096477,-1.437180,...,1.460518,1.572004,1.308313,1.408181,0.205041,0.117051,0.025857,0.080689,0.311381,0.190822
1,Alcohol,付瑞晗,2,0.599527,1.112453,0.374152,0.599123,0.947222,0.145544,2.105166,...,2.053216,5.278246,1.767573,4.543939,0.524562,0.530528,0.348305,0.276095,0.214957,0.148971
2,Alcohol,付瑞晗,3,0.133105,1.244384,0.784893,-0.469845,1.081188,1.044413,-4.690412,...,1.058269,0.795306,0.601766,0.452237,-0.258908,-0.125163,0.260005,0.177508,0.259267,0.124151
3,Alcohol,付瑞晗,4,1.249114,1.324410,0.054951,0.609332,0.421649,-0.501940,0.289411,...,-5.516007,0.603882,-8.017515,0.877743,-0.159490,0.392523,0.287928,0.244415,0.335461,0.253055
4,Alcohol,付瑞晗,5,1.462762,1.520320,0.673478,1.427771,1.266134,0.303502,0.808523,...,0.654377,1.452079,0.224516,0.498206,0.314410,0.028311,0.420035,0.136193,0.296648,0.206192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,Control,陈妍,3,0.480506,0.781104,0.844972,0.065712,0.332182,0.954549,6.201824,...,10.506800,8.394237,10.802331,8.630346,-0.332961,0.381392,0.300467,0.223729,0.435934,-0.002783
220,Control,陈妍,4,0.858444,0.287205,0.201724,0.740466,0.307619,0.352925,2.578917,...,-0.459137,1.321222,0.309093,-0.889454,0.232564,0.143382,0.240019,-0.145777,0.335301,0.145758
221,Control,陈妍,5,0.638032,0.071052,0.083395,0.153254,0.075354,0.068925,2.762618,...,-0.275669,-0.242793,1.219859,1.074378,0.605060,0.605060,0.605060,0.605060,0.605060,0.605060
222,Control,陈妍,6,0.810403,0.183523,0.279126,0.459459,0.097723,0.332291,2.576446,...,-1.902672,1.656347,1.280889,-1.115062,-0.046973,0.166768,0.254342,0.268078,0.213953,0.045828


In [3]:
df.columns

Index(['组别', '受试者', 'session', 'EEG_beta_high_Cz_mean',
       'EEG_beta_high_Fz_mean', 'EEG_beta_high_Pz_mean',
       'EEG_beta_low_Cz_mean', 'EEG_beta_low_Fz_mean', 'EEG_beta_low_Pz_mean',
       'Cz_alpVShi', 'Cz_alpVSlow', 'Cz_theVShi', 'Cz_theVSlow', 'Fz_alpVShi',
       'Fz_alpVSlow', 'Fz_theVShi', 'Fz_theVSlow', 'Pz_alpVShi', 'Pz_alpVSlow',
       'Pz_theVShi', 'Pz_theVSlow', 'DE_Cz_beta_high', 'DE_Cz_beta_low',
       'DE_Fz_beta_high', 'DE_Fz_beta_low', 'DE_Pz_beta_high',
       'DE_Pz_beta_low'],
      dtype='object')

In [5]:
metrics = ['EEG_beta_high_Cz_mean',
           'EEG_beta_high_Fz_mean', 'EEG_beta_high_Pz_mean',
           'EEG_beta_low_Cz_mean', 'EEG_beta_low_Fz_mean', 'EEG_beta_low_Pz_mean',
           'Cz_alpVShi', 'Cz_alpVSlow', 'Cz_theVShi', 'Cz_theVSlow', 'Fz_alpVShi',
           'Fz_alpVSlow', 'Fz_theVShi', 'Fz_theVSlow', 'Pz_alpVShi', 'Pz_alpVSlow',
           'Pz_theVShi', 'Pz_theVSlow', 'DE_Cz_beta_high', 'DE_Cz_beta_low',
           'DE_Fz_beta_high', 'DE_Fz_beta_low', 'DE_Pz_beta_high',
           'DE_Pz_beta_low']
results = {}
for metric in metrics:
    print(f"\n=== 混合效应模型结果：{metric} ===")

    model = smf.mixedlm(
        f"{metric} ~ 组别 * session",
        df,  # 直接用完整数据
        groups=df["受试者"]
    )
    result = model.fit(method="powell", maxiter=500)
    results[metric] = result

    print(result.summary())


=== 混合效应模型结果：EEG_beta_high_Cz_mean ===
               Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: EEG_beta_high_Cz_mean
No. Observations: 224     Method:             REML                 
No. Groups:       32      Scale:              669.4057             
Min. group size:  7       Log-Likelihood:     -1038.6927           
Max. group size:  7       Converged:          Yes                  
Mean group size:  7.0                                              
-------------------------------------------------------------------
                          Coef. Std.Err.   z   P>|z|  [0.025 0.975]
-------------------------------------------------------------------
Intercept                 0.357    5.467 0.065 0.948 -10.357 11.072
组别[T.Control]             4.933    7.731 0.638 0.523 -10.220 20.085
session                   0.080    1.222 0.066 0.948  -2.315  2.476
组别[T.Control]:session     0.199    1.729 0.115 0.908  -3.189  3.588
Group Var              

In [ ]:
def extract_result(result, metric_name):
    summary_df = pd.DataFrame({
        "coef": result.params,
        "std_err": result.bse,
        "z_value": result.tvalues,
        "p_value": result.pvalues
    })
    summary_df["显著性"] = summary_df["p_value"].apply(
        lambda p: "显著" if p < 0.05 else "不显著"
    )
    summary_df["指标"] = metric_name
    summary_df["效应项"] = summary_df.index
    return summary_df.reset_index(drop=True)


# ---------------------------
# 5. 汇总所有模型结果
# ---------------------------
df_list = []
for metric in metrics:
    df_metric = extract_result(results[metric], metric)
    df_list.append(df_metric)

df_all = pd.concat(df_list, axis=0)
df_sig = df_all[df_all["显著性"] == "显著"].copy()

# ---------------------------
# 6. 保存到 Excel
# ---------------------------
with pd.ExcelWriter("混合效应模型结果_EEG_24指标.xlsx") as writer:
    for metric in metrics:
        extract_result(results[metric], metric).to_excel(
            writer, sheet_name=metric[:31], index=False
        )

    df_all.to_excel(writer, sheet_name="全部结果汇总", index=False)
    df_sig.to_excel(writer, sheet_name="显著结果汇总", index=False)

print("✅ 混合效应模型结果保存成功！")